In [0]:
storage_account_name = "librarydatastorage2"
storage_account_key = "<YOUR_STORAGE_KEY>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
train_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/train.csv"
train_df = spark.read.csv(train_path, header=True, inferSchema=True)

In [0]:
train_df.printSchema()
train_df.show(5)

root
 |-- building_id: integer (nullable = true)
 |-- meter: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- meter_reading: double (nullable = true)

+-----------+-----+-------------------+-------------+
|building_id|meter|          timestamp|meter_reading|
+-----------+-----+-------------------+-------------+
|          0|    0|2016-01-01 00:00:00|          0.0|
|          1|    0|2016-01-01 00:00:00|          0.0|
|          2|    0|2016-01-01 00:00:00|          0.0|
|          3|    0|2016-01-01 00:00:00|          0.0|
|          4|    0|2016-01-01 00:00:00|          0.0|
+-----------+-----+-------------------+-------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, to_timestamp

train_df.select([
    (col(c).isNull().cast("int")).alias(c) for c in train_df.columns
]).groupBy().sum().show()

train_df_clean = train_df.dropna()

train_df_clean = train_df_clean.withColumn(
    "timestamp", to_timestamp("timestamp")
)

+----------------+----------+--------------+------------------+
|sum(building_id)|sum(meter)|sum(timestamp)|sum(meter_reading)|
+----------------+----------+--------------+------------------+
|               0|         0|             0|                 0|
+----------------+----------+--------------+------------------+



In [0]:
train_df_clean.printSchema()
train_df_clean.show(5)

root
 |-- building_id: integer (nullable = true)
 |-- meter: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- meter_reading: double (nullable = true)

+-----------+-----+-------------------+-------------+
|building_id|meter|          timestamp|meter_reading|
+-----------+-----+-------------------+-------------+
|          0|    0|2016-01-01 00:00:00|          0.0|
|          1|    0|2016-01-01 00:00:00|          0.0|
|          2|    0|2016-01-01 00:00:00|          0.0|
|          3|    0|2016-01-01 00:00:00|          0.0|
|          4|    0|2016-01-01 00:00:00|          0.0|
+-----------+-----+-------------------+-------------+
only showing top 5 rows


In [0]:
output_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/processed/train_cleaned"

train_df_clean.write.mode("overwrite").parquet(output_path)